已连接到 himga (Python 3.10.18)

# HiMGA基座使用指南

In [3]:
# Imports
import json
import pickle as pkl
from pathlib import Path

from himga.agent import BaseAgent
from himga.data import load_dataset
from himga.eval import compute_metrics, run_eval
from himga.eval.judge import batch_judge
from himga.llm import get_client
from himga.memory import NullMemory

## 加载数据集

In [4]:
# Load datasets
locomo_samples = load_dataset("locomo")
lme_samples = load_dataset("longmemeval", limit=10)

print(f"LoCoMo: {len(locomo_samples)} samples")
print(f"LongMemEval: {len(lme_samples)} samples")

0424-18:03|INFO| [locomo] found at /Volumes/itgz/datasets/locomo
0424-18:03|SUCCESS| Dataset 'locomo' loaded! (10 samples)
0424-18:03|INFO| [longmemeval] found at /Volumes/itgz/datasets/longmemeval
0424-18:03|SUCCESS| Dataset 'longmemeval' loaded! (10 samples)


LoCoMo: 10 samples
LongMemEval: 10 samples


# 查看数据集

In [5]:
# Inspect sessions of a single LoCoMo sample
sample_idx = 1

for ses_idx, ses in enumerate(locomo_samples[sample_idx].sessions):
    print(f"📝Session {ses_idx} date: {ses.date}")
    for msg in ses.messages:
        print(f"{msg.role}: {msg.content}")
    print("💩" * 40)

📝Session 0 date: 2023-01-20 16:04:00
Gina: Hey Jon! Good to see you. What's up? Anything new?
Jon: Hey Gina! Good to see you too. Lost my job as a banker yesterday, so I'm gonna take a shot at starting my own business.
Gina: Sorry about your job Jon, but starting your own business sounds awesome! Unfortunately, I also lost my job at Door Dash this month. What business are you thinking of?
Jon: Sorry to hear that! I'm starting a dance studio 'cause I'm passionate about dancing and it'd be great to share it with others.
Gina: That's cool, Jon! What got you into this biz?
Jon: I've been into dancing since I was a kid and it's been my passion and escape. I wanna start a dance studio so I can teach others the joy that dancing brings me.
Gina: Wow Jon, same here! Dance is pretty much my go-to for stress relief. Got any fave styles?
Jon: Cool, Gina! I love all dances, but contemporary is my top pick. It's so expressive and powerful! What's your fave?
Gina: Yeah, me too! Contemporary dance is 

In [6]:
# Inspect QA pairs of the same sample
for qa in locomo_samples[sample_idx].qa_pairs:
    print(f"QAPair {qa.question_id}: type={qa.question_type}")
    print(f"Question: {qa.question}")
    print(f"Evidence: {qa.evidence}")
    print(f"Answer: {qa.answer}")
    print("💩" * 40)

QAPair 0: type=temporal
Question: When Jon has lost his job as a banker?
Evidence: EvidenceRef(turn_ids=['D1:2'], session_ids=[])
Answer: 19 January, 2023
💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩
QAPair 1: type=temporal
Question: When Gina has lost her job at Door Dash?
Evidence: EvidenceRef(turn_ids=['D1:3'], session_ids=[])
Answer: January, 2023
💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩
QAPair 2: type=open_domain
Question: How do Jon and Gina both like to destress?
Evidence: EvidenceRef(turn_ids=['D1:7', 'D1:6'], session_ids=[])
Answer: by dancing
💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩
QAPair 3: type=single_hop
Question: What do Jon and Gina both have in common?
Evidence: EvidenceRef(turn_ids=['D1:2', 'D1:3', 'D1:4', 'D2:1'], session_ids=[])
Answer: They lost their jobs and decided to start their own businesses.
💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩💩
QAPair 4: type=open_domain
Question: Why did Jon decide to start his dance studio?
Evidence: EvidenceRef(turn_ids=['D1:2', 'D1:4'], 

## 建立Memory
himga内置无记忆Memory，`NullMemeory`

作为开发者，主要任务是建立自己的Memory类
Memory类需要实现以下接口：
- `ingest(message:Message, session: Session)`: 将一个Message中的信息存储到Memory中
- `retrieve(query: str) -> str`: 根据一个查询字符串，从Memory中检索出证据
- `reset()`: 重置Memory,因为benchmark是多轮的，每个轮次需要重置记忆


In [7]:
# Smoke-test NullMemory ingest & retrieve
mem = NullMemory()
mem.ingest(
    session=locomo_samples[sample_idx].sessions[0],
    message=locomo_samples[sample_idx].sessions[0].messages[0],
)
ans = mem.retrieve(query="What did the user say?")
print(f"Retrieved answer: {ans}")

Retrieved answer: 


## 建立Agent

这是HiMGA的自动测评智能体，给他`llm`客户端与`Memory`，他就就可以适配各类评估任务了

In [8]:
# Build agent
llm = get_client(provider="openai", base_url="https://www.dmxapi.cn/v1", batch_size=5)
agent = BaseAgent(memory=mem, llm=llm)
agent.answer(question=locomo_samples[sample_idx].qa_pairs[0].question)

"I'm sorry, but I don't have any information about Jon losing his job as a banker. If you can provide more context or details, I would be happy to help!"

## 回答数据集QAPair
使用如下：
`run_eval(samples, agent)`
便会自动`ingest`数据集中的每个Message，并根据QAPair中的问题进行检索，并在最后给出生成的回答，即如下的`results`

> 这里为了不浪费api 使用假数据

In [9]:
# Run evaluation (cached)
_cache = Path("outputs/locomo_results.pkl")
if _cache.exists():
    with open(_cache, "rb") as f:
        results = pkl.load(f)
else:
    results = run_eval([locomo_samples[0]], agent, show_progress=True)
    with open(_cache, "wb") as f:
        pkl.dump(results, f)

## 指标计算

In [10]:
# LLM-based judging (cached)
llm_scores = batch_judge(
    results,
    llm=llm,
    cache_path=Path("outputs/locomo_judge.json"),
)

In [12]:
# Compute & save metrics
metrics = compute_metrics(
    results,
    metrics=(
        "judge_score",  # -> 只有他要钱
        "exact_match",
        "f1",
        "rouge1",
        "rouge2",
        "rougeL",
        "bleu1",
        "bleu2",
        "bleu4",
        "meteor",
        "bert_f1",
        "sbert_similarity",
    ),
    llm=llm,  # 计算上述judge_score
    cache_path=Path("outputs/locomo_judge.json"),  # cache上述judge_score
)
with open("outputs/locomo_metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 7673.60it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7131.40it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                   

In [13]:
# Report metrics as DataFrame
def report_metrics(metrics):
    import pandas as pd

    rows = {"overall": metrics["overall"]}
    rows.update(metrics["by_type"])
    df = pd.DataFrame(rows).T
    return df


report_metrics(metrics)

,judge_score,exact_match,f1,rouge1,rouge2,rougeL,bleu1,bleu2,bleu4,meteor,bert_f1,sbert_similarity,count
overall,0.0,0.0,0.089080,0.098035,0.016014,0.087278,0.062554,0.023778,0.009460,0.089977,0.854517,0.290389,NaN
temporal,0.0,0.0,0.072202,0.071347,0.000000,0.071347,0.045082,0.012603,0.007239,0.015861,0.841436,0.198815,37.0
multi_hop,0.0,0.0,0.043111,0.057501,0.013002,0.045945,0.040124,0.016443,0.005136,0.066298,0.840216,0.326408,13.0
single_hop,0.0,0.0,0.033047,0.037813,0.003110,0.033714,0.038091,0.011619,0.005094,0.029035,0.836200,0.245667,32.0
open_domain,0.0,0.0,0.111057,0.128565,0.025140,0.110462,0.080745,0.032296,0.011676,0.131114,0.865292,0.323502,70.0
adversarial,0.0,0.0,0.120500,0.125787,0.024649,0.113190,0.072076,0.030195,0.012076,0.135096,0.865191,0.333646,47.0
